[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/mechanical_work/mechanical_work.ipynb)

# Mechanical Work in a Mirror

This example computes the mechanical power delivered by a tangential force to a turning crank and to its mirror image. Mechanical work measures the energy transferred as a force drives a motion, and power is the rate at which that work is done. Reflecting the crank and the applied force in a mirror reverses the sense of rotation, yet the reflected force still pushes along the handle's motion and delivers the same positive power.

In planar projective geometric algebra, the handle position is modelled as a point and the force as an ideal point at infinity, both antivectors with ***intrinsic*** orientation, while the crank's turning rate is modelled as a bivector with ***extrinsic*** orientation. Joining the handle with the applied force forms the intrinsic forque `handle & force` along the line of action, the handle velocity follows from the commutator `handle.commutator(rate)`, and pairing the extrinsic rate with the intrinsic forque, `rate & forque`, yields the power as an extrinsic scalar. Because one reflection `mirror >> x` transforms every element according to its own orientation, reflecting the inputs transforms the forque and the rate together and leaves `rate & forque` unchanged.

In Gibbs–Heaviside vector notation, shifting to the pivot $\mathbf r_0$ gives the handle velocity $\mathbf v = \boldsymbol\omega \times (\mathbf r - \mathbf r_0)$, the torque $\boldsymbol\tau = (\mathbf r - \mathbf r_0) \times \mathbf F$, and the power $P = \boldsymbol\omega \cdot \boldsymbol\tau = \mathbf F \cdot \mathbf v$; under an orthogonal reflection $M$, the polar vectors transform as $M(\mathbf r - \mathbf r_0)$ and $M\mathbf F$, whereas the axial vectors $\boldsymbol\omega$ and $\boldsymbol\tau$ carry an explicit factor of $\det M$ as $(\det M)\,M\boldsymbol\omega$ and $(\det M)\,M\boldsymbol\tau$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA2D
from examples.util.animation import show_animation
from examples.orientation.mechanical_work import render

np.set_printoptions(precision=3, suppress=True)

ga = PGA2D
mv = NumpyContext(ga).multivector
# Trailing type comments mark + extrinsic and - intrinsic orientation.

## Power delivered to a crank and its mirror image

As each crank turns at a steady rate around its pivot, a tangential push turns with the handle and drives the motion along its circular path. Reflecting the pivot, initial handle, applied force, and turning rate in a mirror plane produces the opposite sense of rotation while pairing the reflected rate with the reflected forque, `rate & forque`, into the same positive scalar power.

In [ ]:
separation = 1.65
radius = 0.9
speed = 1
force_strength = 1.5
n_steps = 72
duration_ms = 40
times = np.linspace(0, 2 * np.pi / speed, n_steps, endpoint=False)

In [ ]:
mirror = mv.x                                               # [] Plane+
pivot = mv.point([-separation, 0])                          # [] Point-
initial_handle = mv.point([-separation + radius, 0])        # [] Point-
initial_force = mv.direction([0, force_strength])           # [] Point-

# Two orthogonal planes through the pivot meet along its rotation axis.
rate = ((mv.x + separation * mv.w) ^ mv.y) * speed          # [] Rate+

# Reflect each physical input in the mirror before evolving both cranks.
pivots = stack([pivot, mirror >> pivot])                    # [2] Point-
initial_handles = stack([initial_handle, mirror >> initial_handle])  # [2] Point-
initial_forces = stack([initial_force, mirror >> initial_force])     # [2] Point-
rates = stack([rate, mirror >> rate])                       # [2] Rate+

poses = (rates[None, :] * (-times[:, None] / 2)).exp()      # [n_steps, 2] Motor+
handles = poses >> initial_handles                          # [n_steps, 2] Point-
forces = poses >> initial_forces                            # [n_steps, 2] Point-

# Each handle's velocity follows from its commutator with the turning rate, and joining handle to force forms the forque.
velocities = handles.commutator(rates)                      # [n_steps, 2] Point-
forques = handles & forces                                  # [n_steps, 2] Forque-
powers = rates & forques                                    # [n_steps, 2] Scalar+

print("Crank power", powers[0, 0].coefficient_table(), sep="\n")
print("Mirrored crank power", powers[0, 1].coefficient_table(), sep="\n")
show_animation(render.work_animation(pivots, handles, velocities, forces), "mechanical_work", duration_ms);

In the animation, the dashed vertical line marks the mirror plane, faint grey circles trace the closed handle paths around the dark pivots, teal arrows show the handle velocities, and orange arrows show the applied forces. On the left crank the motion is anticlockwise and on the right crank it is clockwise, while in every frame the orange force arrow points along the teal velocity arrow so both cranks receive the same positive power.

In [ ]:
# checks
expected_power = radius * force_strength * speed
forque_difference = forques[:, 1] - (mirror >> forques[:, 0])               # [n_steps] Forque-
assert (forque_difference.norm() + forque_difference.antinorm() < 1e-12).all()
assert ((velocities[:, 1] - (mirror >> velocities[:, 0])).antinorm() < 1e-12).all()
assert ((powers - expected_power).abs() < 1e-12).all()
assert (((mirror >> powers[:, 0]) - powers[:, 1]).abs() < 1e-12).all()